# Joshi Part 7: Binomial Trees & Implied Volatility (Python)

Binomial tree pricing, American options, and implied volatility via Newton-Raphson.

## 1. CRR Binomial Tree

In [ ]:
call_payoff = lambda s: max(s - 100, 0)
put_payoff = lambda s: max(100 - s, 0)

for steps in [50, 100, 200, 500]:
    c = binomial_european(call_payoff, 100, 0.05, 0.2, 1.0, steps)
    p = binomial_european(put_payoff, 100, 0.05, 0.2, 1.0, steps)
    print(f"Steps={steps:>4}: Call={c:.4f}, Put={p:.4f}")

print(f"\nBS exact: Call=10.4506, Put=5.5735")

## 2. American Put Premium

In [ ]:
steps = 500
euro_put = binomial_european(put_payoff, 100, 0.05, 0.2, 1.0, steps)
amer_put = binomial_american(put_payoff, 100, 0.05, 0.2, 1.0, steps)

print(f"European Put: {euro_put:.4f}")
print(f"American Put: {amer_put:.4f}")
print(f"Early exercise premium: {amer_put - euro_put:.4f}")

## 3. Newton-Raphson Implied Volatility

In [ ]:
# Round-trip test: price at vol=25%, then recover
true_vol = 0.25
price = bs_call(100, 100, 0.05, true_vol, 1.0)
implied, iters = implied_vol_newton(price, 100, 100, 0.05, 1.0)
print(f"True vol:    {true_vol:.6f}")
print(f"Implied vol: {implied:.6f}")
print(f"Iterations:  {iters}")
print(f"Error:       {abs(implied - true_vol):.2e}")

## 4. Implied Volatility Smile

In [ ]:
# Generate a skewed vol smile
S, r, T = 100, 0.05, 1.0
strikes = np.linspace(80, 120, 9)

# Simulate market prices with a skew (lower strike → higher vol)
true_vols = 0.20 + 0.001 * (100 - strikes)  # simple skew
market_prices = [bs_call(S, K, r, v, T) for K, v in zip(strikes, true_vols)]

# Recover implied vols
print(f"{'Strike':>8} {'MktPrice':>10} {'TrueVol':>10} {'ImplVol':>10}")
print("-" * 42)
for K, mp, tv in zip(strikes, market_prices, true_vols):
    iv, _ = implied_vol_newton(mp, S, K, r, T)
    print(f"{K:>8.1f} {mp:>10.4f} {tv:>10.4f} {iv:>10.4f}")